# 📊 EuroSAT DOFA Training — MLflow Instrumented
### Experiment tracking, model registry, and run comparison

This notebook adds full **MLflow** instrumentation to the DOFA EuroSAT
segmentation training pipeline. Every run is tracked, compared, and the
best checkpoint is registered as a versioned model.

---
**What MLflow adds:**
- Automatic logging of all hyperparameters
- Per-epoch metrics (train loss, val loss, val mIoU)
- Training curves saved as artifacts
- Best checkpoint registered in the Model Registry
- Run comparison across experiments in the UI


## ⚙️ Step 1 — Install & Configure MLflow

In [17]:
# Install MLflow if not already installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "mlflow", "--quiet"], check=True)

import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
from pathlib import Path

# ── MLflow configuration ──────────────────────────────────────────────────────
# Tracking URI — local folder (no server needed)
# To use the UI: run `mlflow ui` in your terminal, then open http://127.0.0.1:5000
MLFLOW_TRACKING_URI = "mlruns"
EXPERIMENT_NAME     = "dofa-eurosat-segmentation"
MODEL_NAME          = "dofa-eurosat-segmenter"   # registered model name

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

print("MLflow configured")
print(f"  Tracking URI : {mlflow.get_tracking_uri()}")
print(f"  Experiment   : {EXPERIMENT_NAME}")
print(f"  UI command   : mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print(f"  UI URL       : http://127.0.0.1:5000")


MLflow configured
  Tracking URI : mlruns
  Experiment   : dofa-eurosat-segmentation
  UI command   : mlflow ui --backend-store-uri mlruns
  UI URL       : http://127.0.0.1:5000


## 🧠 Step 2 — Define Hyperparameters

In [18]:
import sys
import torch
import segmentation_models_pytorch as smp

# Add geo-deep-learning to path
sys.path.insert(0, r"C:\geo-deep-learning")
from geo_deep_learning.tasks_with_models.segmentation_dofa import SegmentationDOFA

# ── All hyperparameters in one dict — logged to MLflow automatically ──────────
HPARAMS = {
    # Data
    "dataset":          "EuroSAT-RGB",
    "num_classes":      10,
    "patch_size":       64,
    "batch_size":       16,
    "val_split":        0.2,
    "augmentation":     True,

    # Model
    "encoder":          "vit_base_patch16_224",
    "pretrained":       True,
    "freeze_encoder":   True,

    # Training
    "max_epochs":       10,
    "learning_rate":    1e-4,
    "weight_decay":     1e-4,
    "loss":             "DiceLoss",
    "optimizer":        "Adam",
    "scheduler":        "CosineAnnealingLR",

    # Sentinel-2 wavelengths (RGB bands)
    "wavelength_R":     0.665,
    "wavelength_G":     0.549,
    "wavelength_B":     0.481,

    # Hardware
    "device":           "cuda" if torch.cuda.is_available() else "cpu",
}

CLASS_NAMES = [
    "Annual Crop", "Forest", "Herbaceous Veg", "Highway", "Industrial",
    "Pasture", "Permanent Crop", "Residential", "River", "Sea / Lake",
]

print("Hyperparameters defined:")
for k, v in HPARAMS.items():
    print(f"  {k:25s}: {v}")


Hyperparameters defined:
  dataset                  : EuroSAT-RGB
  num_classes              : 10
  patch_size               : 64
  batch_size               : 16
  val_split                : 0.2
  augmentation             : True
  encoder                  : vit_base_patch16_224
  pretrained               : True
  freeze_encoder           : True
  max_epochs               : 10
  learning_rate            : 0.0001
  weight_decay             : 0.0001
  loss                     : DiceLoss
  optimizer                : Adam
  scheduler                : CosineAnnealingLR
  wavelength_R             : 0.665
  wavelength_G             : 0.549
  wavelength_B             : 0.481
  device                   : cpu


## 📦 Step 3 — Load Model & Checkpoint

In [19]:
# ── Load existing checkpoint ──────────────────────────────────────────────────
CHECKPOINT_PATH = Path(
    r"C:\geo-deep-learning\logs\gdl_experiment\version_11\checkpoints"
    r"\model-epoch=00-val_loss=0.141.ckpt"
)

print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")

ckpt  = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
hp    = ckpt["hyper_parameters"]

model = SegmentationDOFA(
    encoder       = hp["encoder"],
    pretrained    = False,
    image_size    = tuple(hp["image_size"]),
    num_classes   = hp["num_classes"],
    max_samples   = hp.get("max_samples", 2),
    loss          = smp.losses.DiceLoss(mode="multiclass", from_logits=True),
    class_labels  = hp.get("class_labels"),
    class_colors  = hp.get("class_colors"),
    freeze_layers = hp.get("freeze_layers"),
)
model.configure_model()
state = {k.removeprefix("model."): v for k, v in ckpt["state_dict"].items()}
model.model.load_state_dict(state, strict=True)
model.eval()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded: {total/1e6:.1f}M total params, {trainable/1e6:.1f}M trainable")


Checkpoint exists: True


\\?\C:\Users\Admin\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.


Model loaded: 140.3M total params, 35.0M trainable


## 📊 Step 4 — MLflow Instrumented Training Loop

This is the core MLflow integration. Everything below is what you would add
to any existing training loop — it wraps your training in `mlflow.start_run()`
and logs metrics at each epoch.


In [20]:
import numpy as np
import time
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.transforms as T
from PIL import Image

# ── Synthetic dataset for demonstration ───────────────────────────────────────
# Replace with your real EuroSAT DataLoader
def make_synthetic_loader(n=200, batch_size=16, split="train"):
    np.random.seed(42 if split == "train" else 99)
    imgs   = torch.rand(n, 3, 64, 64)
    labels = torch.randint(0, 10, (n, 64, 64))
    ds     = TensorDataset(imgs, labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=(split=="train"))

train_loader = make_synthetic_loader(200, HPARAMS["batch_size"], "train")
val_loader   = make_synthetic_loader(50,  HPARAMS["batch_size"], "val")
print(f"Loaders ready — train: {len(train_loader)} batches  val: {len(val_loader)} batches")

# ── Loss & optimiser ──────────────────────────────────────────────────────────
criterion = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
optimiser = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=HPARAMS["learning_rate"],
    weight_decay=HPARAMS["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=HPARAMS["max_epochs"]
)

device = torch.device(HPARAMS["device"])
model  = model.to(device)
wv     = torch.tensor([[HPARAMS["wavelength_R"],
                         HPARAMS["wavelength_G"],
                         HPARAMS["wavelength_B"]]]).to(device)

def compute_miou(preds, labels, num_classes=10):
    ious = []
    for cls in range(num_classes):
        pred_mask  = (preds == cls)
        label_mask = (labels == cls)
        inter = (pred_mask & label_mask).sum().float()
        union = (pred_mask | label_mask).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return float(np.mean(ious)) if ious else 0.0

print("Training setup complete")


Loaders ready — train: 13 batches  val: 4 batches
Training setup complete


In [21]:
# ── MLflow Training Run ───────────────────────────────────────────────────────

RUN_NAME = "dofa-eurosat-v1"   # Change for each experiment variant

with mlflow.start_run(run_name=RUN_NAME) as run:

    run_id = run.info.run_id
    print(f"MLflow run started: {run_id}")
    print(f"Run name: {RUN_NAME}")

    # ── Log all hyperparameters ───────────────────────────────────────────────
    mlflow.log_params(HPARAMS)
    mlflow.log_param("total_params_M",     round(total/1e6, 1))
    mlflow.log_param("trainable_params_M", round(trainable/1e6, 1))
    mlflow.log_param("train_batches",      len(train_loader))
    mlflow.log_param("val_batches",        len(val_loader))
    print("Hyperparameters logged")

    # ── Training loop ─────────────────────────────────────────────────────────
    history = {"train_loss": [], "val_loss": [], "val_miou": [], "lr": []}
    best_val_loss = float("inf")
    best_epoch    = 0

    for epoch in range(HPARAMS["max_epochs"]):
        t_epoch = time.time()
        model.train()
        train_losses = []

        for batch_imgs, batch_labels in train_loader:
            batch_imgs   = batch_imgs.to(device)
            batch_labels = batch_labels.to(device)
            wv_batch     = wv.expand(batch_imgs.size(0), -1)

            optimiser.zero_grad()
            output = model(batch_imgs, wv_batch)
            loss   = criterion(output.out, batch_labels)
            loss.backward()
            optimiser.step()
            train_losses.append(loss.item())

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # Validation
        model.eval()
        val_losses = []
        val_mious  = []
        with torch.no_grad():
            for batch_imgs, batch_labels in val_loader:
                batch_imgs   = batch_imgs.to(device)
                batch_labels = batch_labels.to(device)
                wv_batch     = wv.expand(batch_imgs.size(0), -1)
                output       = model(batch_imgs, wv_batch)
                loss         = criterion(output.out, batch_labels)
                preds        = output.out.argmax(dim=1)
                val_losses.append(loss.item())
                val_mious.append(compute_miou(preds.cpu(), batch_labels.cpu()))

        train_loss = float(np.mean(train_losses))
        val_loss   = float(np.mean(val_losses))
        val_miou   = float(np.mean(val_mious))
        epoch_time = time.time() - t_epoch

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_miou"].append(val_miou)
        history["lr"].append(current_lr)

        # ── Log per-epoch metrics to MLflow ──────────────────────────────────
        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_miou":   val_miou,
            "lr":         current_lr,
            "epoch_time": epoch_time,
        }, step=epoch)

        # Track best epoch
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            mlflow.log_metric("best_val_loss",  best_val_loss, step=epoch)
            mlflow.log_metric("best_val_miou",  val_miou,      step=epoch)
            mlflow.log_metric("best_epoch",     best_epoch,    step=epoch)

        print(f"  Epoch {epoch+1:02d}/{HPARAMS['max_epochs']}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_miou={val_miou:.4f}  lr={current_lr:.2e}  "
              f"{'* best' if epoch == best_epoch else ''}")

    print(f"\nBest epoch: {best_epoch+1}  val_loss={best_val_loss:.4f}")
    print(f"Run ID: {run_id}")


MLflow run started: 3b98451b4db741809984f6f4f17f64a7
Run name: dofa-eurosat-v1
Hyperparameters logged
  Epoch 01/10  train_loss=0.9016  val_loss=0.9134  val_miou=0.0326  lr=9.76e-05  * best
  Epoch 02/10  train_loss=0.8994  val_loss=0.9017  val_miou=0.0452  lr=9.05e-05  * best
  Epoch 03/10  train_loss=0.8991  val_loss=0.9014  val_miou=0.0485  lr=7.94e-05  * best
  Epoch 04/10  train_loss=0.8983  val_loss=0.9009  val_miou=0.0498  lr=6.55e-05  * best
  Epoch 05/10  train_loss=0.8973  val_loss=0.9016  val_miou=0.0489  lr=5.00e-05  
  Epoch 06/10  train_loss=0.8966  val_loss=0.9033  val_miou=0.0475  lr=3.45e-05  
  Epoch 07/10  train_loss=0.8953  val_loss=0.9037  val_miou=0.0486  lr=2.06e-05  
  Epoch 08/10  train_loss=0.8944  val_loss=0.9025  val_miou=0.0494  lr=9.55e-06  
  Epoch 09/10  train_loss=0.8933  val_loss=0.9028  val_miou=0.0493  lr=2.45e-06  
  Epoch 10/10  train_loss=0.8931  val_loss=0.9018  val_miou=0.0505  lr=0.00e+00  

Best epoch: 4  val_loss=0.9009
Run ID: 3b98451b4db741

## 📈 Step 5 — Log Training Curves as Artifacts

In [5]:
RUN_NAME = "dofa-eurosat-v1"   # must match what you used in the training cell
history  = {
    "train_loss": [0.45, 0.38, 0.32, 0.28, 0.25, 0.23, 0.21, 0.20, 0.19, 0.18],
    "val_loss":   [0.42, 0.35, 0.30, 0.27, 0.24, 0.22, 0.21, 0.20, 0.19, 0.18],
    "val_miou":   [0.31, 0.38, 0.43, 0.47, 0.50, 0.52, 0.54, 0.55, 0.56, 0.57],
    "lr":         [1e-4, 9e-5, 8e-5, 7e-5, 6e-5, 5e-5, 4e-5, 3e-5, 2e-5, 1e-5],
}
best_epoch = 9
HPARAMS    = {"max_epochs": 10}

In [6]:
from pathlib import Path

In [7]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [26]:
import mlflow
from pathlib import Path

mlflow.set_tracking_uri("mlruns")
client     = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("dofa-eurosat-segmentation")
runs       = client.search_runs(experiment.experiment_id, order_by=["start_time DESC"])
run_id     = runs[0].info.run_id
print("Using run_id:", run_id)

Using run_id: 3b98451b4db741809984f6f4f17f64a7


In [8]:
import mlflow
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

mlflow.set_tracking_uri("mlruns")
client     = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("dofa-eurosat-segmentation")
runs       = client.search_runs(experiment.experiment_id, order_by=["start_time DESC"])
run        = runs[0]
run_id     = run.info.run_id

RUN_NAME   = run.data.tags.get("mlflow.runName", "dofa-eurosat-v1")
best_epoch = int(run.data.metrics.get("best_epoch", 0))
HPARAMS    = {"max_epochs": 10}

# Reconstruct history from MLflow metrics
history = {"train_loss": [], "val_loss": [], "val_miou": [], "lr": []}
for metric in ["train_loss", "val_loss", "val_miou", "lr"]:
    points = client.get_metric_history(run_id, metric)
    history[metric] = [p.value for p in sorted(points, key=lambda x: x.step)]

print("Run     :", RUN_NAME)
print("Run ID  :", run_id)
print("Epochs  :", len(history["train_loss"]))

Run     : dofa-eurosat-v1
Run ID  : 3b98451b4db741809984f6f4f17f64a7
Epochs  : 10


In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("DOFA EuroSAT Training — " + RUN_NAME, fontsize=13, fontweight="bold")

epochs = range(1, HPARAMS["max_epochs"] + 1)

axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss", linewidth=2)
axes[0].plot(epochs, history["val_loss"],   "r-o", label="Val Loss",   linewidth=2)
axes[0].axvline(best_epoch+1, color="green", linestyle="--", alpha=0.7, label="Best epoch")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_facecolor("#f8f9fa")

axes[1].plot(epochs, history["val_miou"], "g-o", linewidth=2)
axes[1].axvline(best_epoch+1, color="green", linestyle="--", alpha=0.7)
axes[1].set_title("Val mIoU"); axes[1].set_xlabel("Epoch")
axes[1].grid(alpha=0.3); axes[1].set_facecolor("#f8f9fa")

axes[2].plot(epochs, history["lr"], "m-o", linewidth=2)
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log"); axes[2].grid(alpha=0.3); axes[2].set_facecolor("#f8f9fa")

fig.patch.set_facecolor("white")
plt.tight_layout()

curve_path = Path("outputs/mlflow_training_curves.png")
curve_path.parent.mkdir(exist_ok=True)
fig.savefig(curve_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

# Log the figure as an MLflow artifact
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(curve_path), artifact_path="plots")
    print("Training curves logged as MLflow artifact")


Training curves logged as MLflow artifact


C:\Users\Admin\AppData\Local\Temp\ipykernel_15560\3222209481.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🏆 Step 6 — Register Best Checkpoint in MLflow Model Registry

In [9]:
from mlflow.models import infer_signature
import torch

# ── Log checkpoint file as artifact ──────────────────────────────────────────
with mlflow.start_run(run_id=run_id):
    # Log the checkpoint file itself
    if CHECKPOINT_PATH.exists():
        mlflow.log_artifact(str(CHECKPOINT_PATH), artifact_path="checkpoints")
        print(f"Checkpoint logged: {CHECKPOINT_PATH.name}")

    # Log summary metrics
    mlflow.log_metrics({
        "final_best_val_loss": best_val_loss,
        "final_best_val_miou": history["val_miou"][best_epoch],
        "final_best_epoch":    float(best_epoch + 1),
    })

    # ── Register the model in MLflow Model Registry ───────────────────────────
    # Create a dummy input for signature inference
    dummy_img  = torch.rand(1, 3, 64, 64)
    dummy_wv = torch.tensor([[0.665, 0.549, 0.481]])
    model_cpu  = model.cpu().eval()
    with torch.no_grad():
        dummy_out = model_cpu(dummy_img, dummy_wv)

    # Log model to registry
    model_info = mlflow.pytorch.log_model(
        pytorch_model = model_cpu,
        artifact_path = "model",
        registered_model_name = MODEL_NAME,
    )
    print(f"Model registered as: {MODEL_NAME}")
    print(f"  Model URI: {model_info.model_uri}")
    print(f"  Run ID  : {run_id}")


NameError: name 'CHECKPOINT_PATH' is not defined

## 🔍 Step 7 — Query Run Results Programmatically

In [10]:
# ── List all runs in the experiment ──────────────────────────────────────────
import pandas as pd
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs       = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.best_val_loss ASC"],
)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Total runs: {len(runs)}")
print()

rows = []
for r in runs:
    rows.append({
        "run_name":      r.data.tags.get("mlflow.runName", "unnamed"),
        "run_id":        r.info.run_id[:8] + "...",
        "status":        r.info.status,
        "val_loss":      round(r.data.metrics.get("best_val_loss", float("nan")), 4),
        "val_miou":      round(r.data.metrics.get("best_val_miou", float("nan")), 4),
        "best_epoch":    int(r.data.metrics.get("best_epoch", 0)) + 1,
        "batch_size":    r.data.params.get("batch_size", "?"),
        "lr":            r.data.params.get("learning_rate", "?"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


NameError: name 'MLFLOW_TRACKING_URI' is not defined

## 🔁 Step 8 — How to Run Multiple Experiments

In [11]:
# ── Example: run with different hyperparameters and compare in MLflow UI ──────
# Just change these values and re-run the training cell above

EXPERIMENT_VARIANTS = [
    {"run_name": "dofa-eurosat-lr1e4",  "learning_rate": 1e-4, "batch_size": 16},
    {"run_name": "dofa-eurosat-lr5e5",  "learning_rate": 5e-5, "batch_size": 16},
    {"run_name": "dofa-eurosat-bs32",   "learning_rate": 1e-4, "batch_size": 32},
]

print("To run multiple experiments and compare them in MLflow UI:")
print()
print("1. Change RUN_NAME and any HPARAMS values")
print("2. Re-run Step 4 (the training cell)")
print("3. All runs appear in the MLflow UI automatically")
print()
print("MLflow UI:")
print("  Terminal : mlflow ui --backend-store-uri mlruns")
print("  Browser  : http://127.0.0.1:5000")
print()
print("In the UI you can:")
print("  - Compare runs side by side")
print("  - Plot any metric across runs")
print("  - Download the best checkpoint")
print("  - View the registered model versions")
print()
print("Current experiment runs:")

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs([experiment.experiment_id])
for r in runs:
    name = r.data.tags.get("mlflow.runName", "unnamed")
    loss = r.data.metrics.get("best_val_loss", float("nan"))
    print(f"  {name:30s}  val_loss={loss:.4f}  id={r.info.run_id[:12]}")


To run multiple experiments and compare them in MLflow UI:

1. Change RUN_NAME and any HPARAMS values
2. Re-run Step 4 (the training cell)
3. All runs appear in the MLflow UI automatically

MLflow UI:
  Terminal : mlflow ui --backend-store-uri mlruns
  Browser  : http://127.0.0.1:5000

In the UI you can:
  - Compare runs side by side
  - Plot any metric across runs
  - Download the best checkpoint
  - View the registered model versions

Current experiment runs:


NameError: name 'EXPERIMENT_NAME' is not defined

## ✅ Summary

| What was added | How |
|---|---|
| Hyperparameter logging | `mlflow.log_params(HPARAMS)` |
| Per-epoch metrics | `mlflow.log_metrics({...}, step=epoch)` |
| Best epoch tracking | `mlflow.log_metric("best_val_loss", ...)` |
| Training curves | `mlflow.log_artifact("plots/...")` |
| Checkpoint logging | `mlflow.log_artifact("checkpoints/...")` |
| Model registration | `mlflow.pytorch.log_model(..., registered_model_name=...)` |
| Run comparison | MLflow UI at http://127.0.0.1:5000 |

To view runs:
```bash
mlflow ui --backend-store-uri mlruns
```
